In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

font_path      = 'font-folder/Roboto-Light.ttf'
font_path_bold = 'font-folder/Roboto-Bold.ttf'
font_prop         = fm.FontProperties(fname=font_path,      size=14)
font_prop_small   = fm.FontProperties(fname=font_path,      size=9)
font_prop_numbers = fm.FontProperties(fname=font_path,      size=9)
font_prop_title   = fm.FontProperties(fname=font_path_bold, size=16)

In [ ]:
df_official = pd.read_csv('../model-evaluation/df_surv_flu_trend_cat_CDC_official.csv')
df_official['method'] = 'CDC official'
df_adapted = pd.read_csv('../model-evaluation/df_surv_flu_trend_cat_CDC_adapted.csv')
df_adapted['method'] = 'CDC adapted'
# merge the two dataframes
df_combined = pd.merge(df_official[['season', 'location', 'obs', 'truth_date', 'method', 'trend_cat']],
                       df_adapted[['season', 'location', 'obs', 'truth_date', 'method', 'trend_cat']], on=['season', 'location', 'truth_date'])

In [ ]:
df_combined

In [ ]:
import pandas as pd

# ---------------------------------------------------------
# 1) Matrice di confusione / cross-tabulazione
# ---------------------------------------------------------
confusion = pd.crosstab(df_combined['trend_cat_x'], df_combined['trend_cat_y'], 
                          rownames=['CDC official'], colnames=['CDC adapted'])
print(confusion)

# stessa cosa in percentuale (per riga, così vedi quanto ogni categoria di X viene "confermata" da Y)
confusion_pct = pd.crosstab(df_combined['trend_cat_x'], df_combined['trend_cat_y'], normalize='index') * 100
print(confusion_pct.round(1))


# ---------------------------------------------------------
# 2) Percentuale di accordo semplice (quante volte i due metodi coincidono)
# ---------------------------------------------------------
agreement = (df_combined['trend_cat_x'] == df_combined['trend_cat_y']).mean() * 100
print(f"Percentuale di accordo tra i due metodi: {agreement:.1f}%")


# ---------------------------------------------------------
# 3) Cohen's Kappa — misura di accordo che corregge per il caso
#    (più informativo del semplice % di accordo, soprattutto se una categoria
#    come "stable" è molto più frequente delle altre)
# ---------------------------------------------------------
from sklearn.metrics import cohen_kappa_score

kappa = cohen_kappa_score(df_combined['trend_cat_x'], df_combined['trend_cat_y'])
print(f"Cohen's Kappa: {kappa:.3f}")

# interpretazione standard (Landis & Koch, 1977):
# <0: nessun accordo | 0-0.20: minimo | 0.21-0.40: discreto
# 0.41-0.60: moderato | 0.61-0.80: sostanziale | 0.81-1: quasi perfetto

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patches as mpatches
import matplotlib.dates as mdates

# scegli location
loc = 'US'
df_plot_full = df_combined[df_combined['location'] == loc].copy()
df_plot_full['truth_date'] = pd.to_datetime(df_plot_full['truth_date'])

cat_colors = {
    'large_decrease': '#2166AC',  # blu scuro — declino ripido
    'decrease':        '#92C5DE',  # azzurro — declino lieve
    'stable':          '#B0B0B0',  # grigio — stabile
    'increase':        '#F4A582',  # arancio — crescita lieve
    'large_increase':  '#D6604D',  # rosso — crescita ripida
}

cat_labels_display = {
    'large_decrease': 'Large decrease',
    'decrease':        'Decrease',
    'stable':          'Stable',
    'increase':        'Increase',
    'large_increase':  'Large increase',
}
cat_labels_ordered = ['large_decrease', 'decrease', 'stable', 'increase', 'large_increase']

# configurazione dei 4 pannelli: (riga, colonna) -> (season, colonna_trend, titolo)
panel_config = [
    (0, 0, '2024/2025', 'trend_cat_x', 'A. 2024/25 — CDC official methodology'),
    (0, 1, '2024/2025', 'trend_cat_y', 'B. 2024/25 — CDC adapted methodology'),
    (1, 0, '2025/2026', 'trend_cat_x', 'C. 2025/26 — CDC official methodology'),
    (1, 1, '2025/2026', 'trend_cat_y', 'D. 2025/26 — CDC adapted methodology'),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 8), dpi=200, sharey='row')

for row, col, season, trend_col, title in panel_config:
    ax = axes[row, col]
    df_panel = df_plot_full[df_plot_full['season'] == season]

    # linea di sfondo
    ax.plot(df_panel['truth_date'], df_panel['obs_x'],
            color='#cccccc', linewidth=1.0, zorder=1)

    # punti colorati per categoria di trend
    for cat, color in cat_colors.items():
        sub = df_panel[df_panel[trend_col] == cat]
        ax.scatter(sub['truth_date'], sub['obs_x'],
                   color=color, s=40, zorder=2, label=cat_labels_display[cat])

    ax.set_title(title, fontproperties=font_prop_title, loc='left', x=0.01)

    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right',
             fontproperties=font_prop)
    for lbl in ax.get_yticklabels():
        lbl.set_fontproperties(font_prop)

    # label assi solo su bordo sinistro/basso della griglia
    if col == 0:
        ax.set_ylabel('Influenza hospital admissions', fontproperties=font_prop)
    if row == 1:
        ax.set_xlabel(' ', fontproperties=font_prop)

# legenda unica condivisa, sotto tutti i pannelli
legend_handles = [
    mpatches.Patch(facecolor=cat_colors[cat], label=cat_labels_display[cat])
    for cat in cat_labels_ordered
]
fig.legend(handles=legend_handles, frameon=False, prop=font_prop,
           loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig('../model-evaluation/figures/trend_classification_2x2.png', dpi=300, bbox_inches='tight')
plt.savefig('../model-evaluation/figures/trend_classification_2x2.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
df_plot_full

In [ ]:
import pandas as pd

# ---------------------------------------------------------
# 1) Matrice di confusione / cross-tabulazione
# ---------------------------------------------------------
df_plot_full_2024 = df_plot_full[df_plot_full['season'] == '2024/2025']
confusion = pd.crosstab(df_plot_full_2024['trend_cat_x'], df_plot_full_2024['trend_cat_y'], 
                          rownames=['CDC official'], colnames=['CDC adapted'])
print(confusion)

# stessa cosa in percentuale (per riga, così vedi quanto ogni categoria di X viene "confermata" da Y)
confusion_pct = pd.crosstab(df_plot_full_2024['trend_cat_x'], df_plot_full_2024['trend_cat_y'], normalize='index') * 100
print(confusion_pct.round(1))


# ---------------------------------------------------------
# 2) Percentuale di accordo semplice (quante volte i due metodi coincidono)
# ---------------------------------------------------------
agreement = (df_plot_full_2024['trend_cat_x'] == df_plot_full_2024['trend_cat_y']).mean() * 100
print(f"Percentuale di accordo tra i due metodi: {agreement:.1f}%")


# ---------------------------------------------------------
# 3) Cohen's Kappa — misura di accordo che corregge per il caso
#    (più informativo del semplice % di accordo, soprattutto se una categoria
#    come "stable" è molto più frequente delle altre)
# ---------------------------------------------------------
from sklearn.metrics import cohen_kappa_score

kappa = cohen_kappa_score(df_plot_full_2024['trend_cat_x'], df_plot_full_2024['trend_cat_y'])
print(f"Cohen's Kappa: {kappa:.3f}")

In [ ]:
import pandas as pd

# ---------------------------------------------------------
# 1) Matrice di confusione / cross-tabulazione
# ---------------------------------------------------------
df_plot_full_2025 = df_plot_full[df_plot_full['season'] == '2025/2026']
confusion = pd.crosstab(df_plot_full_2025['trend_cat_x'], df_plot_full_2025['trend_cat_y'], 
                          rownames=['CDC official'], colnames=['CDC adapted'])
print(confusion)

# stessa cosa in percentuale (per riga, così vedi quanto ogni categoria di X viene "confermata" da Y)
confusion_pct = pd.crosstab(df_plot_full_2025['trend_cat_x'], df_plot_full_2025['trend_cat_y'], normalize='index') * 100
print(confusion_pct.round(1))


# ---------------------------------------------------------
# 2) Percentuale di accordo semplice (quante volte i due metodi coincidono)
# ---------------------------------------------------------
agreement = (df_plot_full_2025['trend_cat_x'] == df_plot_full_2025['trend_cat_y']).mean() * 100
print(f"Percentuale di accordo tra i due metodi: {agreement:.1f}%")


# ---------------------------------------------------------
# 3) Cohen's Kappa — misura di accordo che corregge per il caso
#    (più informativo del semplice % di accordo, soprattutto se una categoria
#    come "stable" è molto più frequente delle altre)
# ---------------------------------------------------------
from sklearn.metrics import cohen_kappa_score

kappa = cohen_kappa_score(df_plot_full_2025['trend_cat_x'], df_plot_full_2025['trend_cat_y'])
print(f"Cohen's Kappa: {kappa:.3f}")